# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the mlcroissant dataset
dataset = mlc.Dataset(croissant_url)

# Access and print key metadata (as attributes)
metadata = dataset.metadata
print(f"Dataset: {getattr(metadata, 'name', None)}\n")
print(f"Description: {getattr(metadata, 'description', None)}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs (`@id`).

In [ ]:
# List all record sets and their field and column @ids
print('Record sets found:')
record_sets = list(dataset.record_sets())
for rs in record_sets:
    print(f"- Record set name: {rs.name}")
    print(f"  @id: {rs['@id']}")
    # List fields and columns
    if hasattr(rs, 'fields') and rs.fields:
        print('  Fields:')
        for fld in rs.fields:
            print(f"    - {fld.name} (@id: {fld['@id']})")
    if hasattr(rs, 'columns') and rs.columns:
        print('  Columns:')
        for col in rs.columns:
            print(f"    - {col.name} (@id: {col['@id']})")
    print()

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.

> Note: Use the exact `@id` as found in the "Data Overview" above to reference each record set and column for extraction.

In [ ]:
# Extract data from all record sets and put into Pandas DataFrames
dataframes = {}
record_set_ids = [rs['@id'] for rs in record_sets]

for rs in record_sets:
    rs_id = rs['@id']
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f'Record set {rs_id}: loaded {len(df)} records.')
        else:
            print(f'Record set {rs_id}: no records found.')
    except Exception as e:
        print(f'Failed to load record set {rs_id}: {e}')

# Show the columns of the first non-empty DataFrame
first_df_id = None
for _id, df in dataframes.items():
    if not df.empty:
        first_df_id = _id
        break
if first_df_id:
    print(f'\nColumns in DataFrame for record set {@id}: {first_df_id}')
    print(dataframes[first_df_id].columns.tolist())
    display(dataframes[first_df_id].head())
else:
    print('No dataframes with data available in record sets.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

In [ ]:
# For EDA, pick a record set and a numeric field (by @id)
# Update these IDs as identified in "Data Overview"

# Example record set and field IDs — update as appropriate for your dataset!
chosen_record_set_id = first_df_id  # Choose one with data
df = dataframes[chosen_record_set_id]

# Display available columns and sample values for guidance
print(f"Columns in chosen DataFrame ({chosen_record_set_id}):\n{df.columns.tolist()}")
print("\nSample values:")
display(df.head())

# Choose a numeric field for filtering, e.g., 'cr:log_likelihood' or similar, based on your columns
numeric_field_id = None
for col in df.columns:
    # Try to heuristically select a numeric field (if present)
    if 'log_likelihood' in col or 'coef' in col:
        numeric_field_id = col
        break
if not numeric_field_id:
    # If not found, pick the first field of numeric dtype
    for col in df.select_dtypes(include='number').columns:
        numeric_field_id = col
        break

print(f"\nSelected numeric field: {numeric_field_id}")

if numeric_field_id is not None:
    # We set a threshold around 10 for demo purposes; adapt as needed
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > {threshold} (showing first 5):")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / (filtered_df[numeric_field_id].std() if filtered_df[numeric_field_id].std() else 1)

    print(f"\nNormalized {numeric_field_id} for filtered records (showing first 5):")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try a group by on another field (for demo, pick the first non-numeric field)
    group_field = None
    for col in df.columns:
        if col != numeric_field_id and df[col].dtype == "object":
            group_field = col
            break
    print(f"\nGroup field: {group_field}")
    
    if group_field and group_field in df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean of {numeric_field_id} by {group_field} (showing first 5):")
        display(grouped_df.head())
else:
    print("No numeric field found for filtering and normalization.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here, we will:
- Plot the distribution of the numeric field
- Optionally, show group-wise boxplots if a group field is available

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Check if filtered_df and numeric_field_id are available
if 'filtered_df' in locals() and numeric_field_id is not None and not filtered_df.empty:
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[numeric_field_id], kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field_id} (> {threshold})")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
    
    # If there is a group_field, show a boxplot
    if 'group_field' in locals() and group_field and group_field in filtered_df.columns:
        plt.figure(figsize=(12,4))
        sns.boxplot(data=filtered_df, x=group_field, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print('Not enough data for visualization.')

## 6. Conclusion
This notebook demonstrates step-by-step exploration of the FAIR^2 dataset using the `mlcroissant` library. 

Key findings and observations include:
- Successful load and overview of dataset metadata and record sets using Croissant `@id` references.
- Extraction of data into Pandas DataFrames and demonstration of basic exploratory analysis workflows: filtering, normalization, and grouping.
- Simple visualizations to explore value distributions and groupwise statistics.

For more sophisticated analyses, consult the dataset's documentation and extend this notebook with additional domain-specific logic and visualization.